# 04 — Fresh-entry detection  (the flow engine)
**Supersedes notebook 03 for live trading.**

Notebook 03 looked at a *static snapshot* of who holds what. The live data proved
the flaw: by the time 3 sharps visibly agree, the price has already moved (the Iran
market ran 0.54 → 0.90 before consensus was visible). A snapshot is too late.

This notebook fixes that. Instead of "who holds what", it asks **"who just *entered*,
and can I still get in near their price?"** It reads each roster wallet's recent BUY
activity (last `ENTRY_LOOKBACK_HOURS`), finds markets where **several sharps entered
the same side recently**, and keeps only those where the **current price is still
close to what they paid**.

Run this **daily** (or more often). Requires `roster.json` from notebook 02.


In [1]:
import importlib, pmc, json, time
importlib.reload(pmc)
from pmc import (CFG, get_recent_buys, get_open_positions, portfolio_value,
                 position_size_eur)
import pandas as pd, numpy as np

roster = json.load(open("roster.json"))["wallets"]
skill_by_wallet = {r["wallet"]: r.get("skill", 0.5) for r in roster}
since_ts = int(time.time()) - CFG.ENTRY_LOOKBACK_HOURS * 3600
print(f"{len(roster)} roster wallets | looking back {CFG.ENTRY_LOOKBACK_HOURS}h "
      f"(since unix {since_ts})")

27 roster wallets | looking back 72h (since unix 1782503387)


## 1. Pull recent BUYs + current price for every roster wallet
We join two trusted endpoints by token id (`asset`):
- `/activity` → *when* they bought and at *what price* (the flow).
- `/positions` → the *current* price and their conviction (only if they still hold it).

If a wallet already sold a position, we skip it — their exit means it is no longer a
live entry signal.

In [2]:
rows = []
for r in roster:
    w = r["wallet"]
    buys = get_recent_buys(w, since_ts)
    if not buys:
        continue
    pos = get_open_positions(w)
    pv = portfolio_value(pos) or 1.0
    pos_by_asset = {p.get("asset"): p for p in pos}
    for b in buys:
        held = pos_by_asset.get(b.get("asset"))
        if not held:            # already exited -> not a live signal
            continue
        usdc = float(b.get("usdcSize") or (float(b.get("size") or 0) * float(b.get("price") or 0)))
        rows.append({
            "wallet": w,
            "skill": skill_by_wallet.get(w, 0.5),
            "conditionId": b.get("conditionId"),
            "asset": b.get("asset"),
            "outcome": b.get("outcome"),
            "title": b.get("title"),
            "entry_price": float(b.get("price") or 0),
            "usdc": usdc,
            "ts": int(b.get("timestamp") or 0),
            "cur_price": float(held.get("curPrice") or 0),
            "cur_value": float(held.get("currentValue") or 0),
            "conviction": min(CFG.CONVICTION_CAP, float(held.get("currentValue") or 0) / pv),
        })

buys_df = pd.DataFrame(rows, columns=["wallet","skill","conditionId","asset","outcome",
                                      "title","entry_price","usdc","ts","cur_price",
                                      "cur_value","conviction"]).dropna(subset=["conditionId","outcome"])
print(f"{len(buys_df)} recent BUY rows from roster sharps "
      f"across {buys_df['conditionId'].nunique() if len(buys_df) else 0} markets")

3382 recent BUY rows from roster sharps across 85 markets


## 2. Collapse to one row per wallet per market-side
A sharp may have bought the same side several times in the window — that is one
conviction, not many. We volume-weight their entry price and sum their notional,
then drop dust trades below `MIN_TRADE_USDC`.

In [3]:
def wavg(df, val, wt):
    wsum = df[wt].sum()
    return (df[val] * df[wt]).sum() / wsum if wsum else df[val].mean()

per_wallet = []
for (w, cid, outcome), g in buys_df.groupby(["wallet", "conditionId", "outcome"]):
    usdc = g["usdc"].sum()
    if usdc < CFG.MIN_TRADE_USDC:
        continue
    per_wallet.append({
        "wallet": w, "skill": g["skill"].iloc[0],
        "conditionId": cid, "outcome": outcome, "title": g["title"].iloc[0],
        "entry_price": round(wavg(g, "entry_price", "usdc"), 4),
        "usdc": round(usdc, 2),
        "last_ts": g["ts"].max(),
        "cur_price": round(g["cur_price"].iloc[0], 4),
        "conviction": g["conviction"].iloc[0],
    })
pw = pd.DataFrame(per_wallet)
print(f"{len(pw)} wallet-level fresh entries (>= ${CFG.MIN_TRADE_USDC:.0f})")

64 wallet-level fresh entries (>= $500)


## 3. Group into fresh consensus per (market, outcome)
`score` = sum of backer skill (≈ 2 mid-skill sharps → ~0.8–1.0). `dissent` = roster
sharps who recently bought the *opposite* side of the same market.

In [4]:
sig = []
for (cid, outcome), g in pw.groupby(["conditionId", "outcome"]):
    other = pw[(pw["conditionId"] == cid) & (pw["outcome"] != outcome)]
    sig.append({
        "conditionId": cid,
        "title": g["title"].iloc[0],
        "outcome": outcome,
        "backers": g["wallet"].nunique(),
        "dissenters": other["wallet"].nunique(),
        "score": round(g["skill"].sum(), 3),
        "median_entry": round(g["entry_price"].median(), 3),
        "cur_price": round(g["cur_price"].median(), 3),
        "total_usdc": round(g["usdc"].sum(), 0),
        "newest_hrs_ago": round((time.time() - g["last_ts"].max()) / 3600, 1),
    })
sig_df = pd.DataFrame(sig).sort_values(["backers", "score"], ascending=False)
print(f"{len(sig_df)} fresh (market, outcome) groups")
sig_df.head(20)

64 fresh (market, outcome) groups


,conditionId,title,outcome,backers,dissenters,score,median_entry,cur_price,total_usdc,newest_hrs_ago
2,0x178b29c0bb3a1c482f37f42e2420a1aa686b960ef000...,Will Claude Fable 5 be restored for US custome...,Yes,1,0,0.493,0.316,0.195,1181.0,70.1
3,0x1f5b9dd89471579b5eef1bbee88a966e50a843d7f79e...,Will Manny Rutinel be the Democratic nominee f...,No,1,0,0.493,0.055,0.065,823.0,2.7
9,0x325d2f917428a0d39ac7563770ec519208666b8c9260...,"US x Iran diplomatic meeting by July 17, 2026?",No,1,0,0.493,0.253,0.345,532.0,5.2
10,0x348cd9adf4f6855f58bd9c6dbf9ff251c4142ef77233...,Strait of Hormuz traffic returns to normal by ...,Yes,1,0,0.493,0.049,0.009,1533.0,68.7
21,0x545e1bf6ec42e0e8a776a3c3e11ed225f8d3dfcc74f9...,"US x Iran diplomatic meeting by July 3, 2026?",No,1,0,0.493,0.314,0.665,5861.0,5.2
26,0x79c850d5dd022d89aaf4f92105657d7a2e0518d4e4d4...,"US x Iran diplomatic meeting by July 10, 2026?",No,1,0,0.493,0.268,0.510,553.0,5.2
29,0x80026e4a0a2aad15549a47671a54aa95afc86325c604...,Will Melat Kiros be the Democratic nominee for...,No,1,0,0.493,0.217,0.225,5226.0,0.3
36,0x8e6c5f24da6e0a987c4b5cd73d3b341152ada808416a...,Will Claude Fable 5 be restored for US custome...,Yes,1,0,0.493,0.827,0.837,689.0,70.1
51,0xcacd424332319971356c757c025b45a4eb5643b919fb...,Strait of Hormuz traffic returns to normal by ...,Yes,1,0,0.493,0.195,0.165,5589.0,1.9
54,0xd73237bb27cdc455578e9a0788c358bd79609d394f43...,SCOTUS bars counting mail ballots after electi...,No,1,0,0.493,0.783,0.994,10818.0,5.8


## 4. Gate + price guard (Blueprint §4–§5)

In [5]:
def gate(r):
    price_ok = (
        r["cur_price"] <= r["median_entry"] + CFG.SLIPPAGE_TOLERANCE and
        CFG.PRICE_FLOOR <= r["cur_price"] <= CFG.PRICE_CEILING
    )
    return (
        r["backers"]    >= CFG.MIN_FRESH_BACKERS and
        r["dissenters"] <= CFG.MAX_DISSENTERS and
        r["score"]      >= CFG.FRESH_SCORE_CUTOFF and
        price_ok
    )

live = sig_df[sig_df.apply(gate, axis=1)].copy() if len(sig_df) else sig_df
print(f"{len(live)} fresh-entry signals pass the full gate")

0 fresh-entry signals pass the full gate


## 5. Suggested balanced sizing + save
Same ¼-Kelly logic as the blueprint. A **proposal**, not an order — you execute.

In [6]:
def size_row(r):
    edge_margin = min(0.10, 0.04 * (r["score"] / CFG.FRESH_SCORE_CUTOFF))
    p_est = min(0.95, r["cur_price"] + edge_margin)
    eur = position_size_eur(p_est, r["cur_price"], r["score"], CFG.FRESH_SCORE_CUTOFF)
    return pd.Series({"p_est": round(p_est, 3), "suggested_eur": eur})

cols = ["title","outcome","backers","dissenters","score","median_entry","cur_price",
        "newest_hrs_ago","total_usdc","p_est","suggested_eur"]
if len(live):
    live = pd.concat([live, live.apply(size_row, axis=1)], axis=1)
    cap = CFG.MAX_TOTAL_DEPLOYED_PCT * CFG.BANKROLL
    live["cum"] = live["suggested_eur"].cumsum()
    live["suggested_eur"] = live.apply(lambda r: r["suggested_eur"] if r["cum"] <= cap else 0.0, axis=1)
    from datetime import datetime
    fname = f"fresh_signals_{datetime.now():%Y%m%d_%H%M}.csv"
    live[cols].to_csv(fname, index=False)
    print("saved", fname)
    display_cols = live[cols]
else:
    display_cols = "No fresh-entry signals right now. Re-run later — flow signals come and go."
display_cols

'No fresh-entry signals right now. Re-run later — flow signals come and go.'

---
### Why this is the right lens
The static engine (03) asked *who holds what* and almost always found the price had
already moved. This engine asks *who just moved* and whether you can still follow —
directly targeting the latency problem.

### Honest caveat (still unproven)
Finding fresh overlapping entries is necessary but **not proof of edge**. Two sharps
buying the same thing today does not guarantee it pays. The only way to know is the
**backtest**: replay history, simulate "buy when N sharps entered within the window
near current price", and measure realized return vs. the market. Until then, treat
these as *candidates to study and paper-trade*, not green lights.

### Tuning knobs (in `pmc.py`)
`ENTRY_LOOKBACK_HOURS` (wider window → more signals, more stale), `MIN_FRESH_BACKERS`
(2 is permissive; 3 is stricter), `FRESH_SCORE_CUTOFF`, `MIN_TRADE_USDC`.